In [5]:
import os

# Create directories for the skills and download the SKILL.md files

# Skill: flyrank-data
skill_path_flyrank = 'skills/flyrank/flyrank-data'
os.makedirs(skill_path_flyrank, exist_ok=True)
skill_url_flyrank = 'https://raw.githubusercontent.com/HassanNawaz14/FlyRank-ML-Internship/refs/heads/main/skills/flyrank/flyrank-data/SKILL.md'
!wget -q -O "{skill_path_flyrank}/SKILL.md" "{skill_url_flyrank}"
print(f"Loaded: {skill_path_flyrank}/SKILL.md")

# Skill: building-baselines
skill_path_baselines = 'skills/building-baselines'
os.makedirs(skill_path_baselines, exist_ok=True)
skill_url_baselines = 'https://raw.githubusercontent.com/HassanNawaz14/FlyRank-ML-Internship/refs/heads/main/skills/building-baselines/SKILL.md'
!wget -q -O "{skill_path_baselines}/SKILL.md" "{skill_url_baselines}"
print(f"Loaded: {skill_path_baselines}/SKILL.md")

Loaded: skills/flyrank/flyrank-data/SKILL.md
Loaded: skills/building-baselines/SKILL.md


In [1]:
import os
os.makedirs('skills/directing-your-ai-assistant', exist_ok=True)
os.makedirs('skills/building-baselines', exist_ok=True)
os.makedirs('skills/flyrank/flyrank-data', exist_ok=True)
!wget -q -O "skills/directing-your-ai-assistant/SKILL.md" "https://raw.githubusercontent.com/HassanNawaz14/FlyRank-ML-Internship/refs/heads/main/skills/directing-your-ai-assistant/SKILL.md"
!wget -q -O "skills/building-baselines/SKILL.md" "https://raw.githubusercontent.com/HassanNawaz14/FlyRank-ML-Internship/refs/heads/main/skills/building-baselines/SKILL.md"
!wget -q -O "skills/flyrank/flyrank-data/SKILL.md" "https://raw.githubusercontent.com/HassanNawaz14/FlyRank-ML-Internship/refs/heads/main/skills/flyrank/flyrank-data/SKILL.md"
print("skills loaded")

skills loaded


In [2]:
import duckdb
import pandas as pd
import os

HF_TOKEN = os.getenv('HF_TOKEN')
if HF_TOKEN is None:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except:
        pass

if HF_TOKEN is None:
    from getpass import getpass
    HF_TOKEN = getpass('Enter your Hugging Face token: ')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':    f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':    f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':     f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
}
print(con.execute(f"SELECT COUNT(*) FROM {TABLES['dim_clients']}").fetchone())

(104,)


In [4]:
print("dim_content columns:")
print(con.execute(f"DESCRIBE SELECT * FROM {TABLES['dim_content']}").fetchdf().to_string())
print("\ndim_clients columns:")
print(con.execute(f"DESCRIBE SELECT * FROM {TABLES['dim_clients']}").fetchdf().to_string())

dim_content columns:
                   column_name column_type null   key default extra
0               client_hash_id     VARCHAR  YES  None    None  None
1              content_hash_id     VARCHAR  YES  None    None  None
2              keyword_hash_id     VARCHAR  YES  None    None  None
3                  url_hash_id     VARCHAR  YES  None    None  None
4           keyword_char_count      BIGINT  YES  None    None  None
5          keyword_token_count      BIGINT  YES  None    None  None
6               url_char_count      BIGINT  YES  None    None  None
7         content_created_date        DATE  YES  None    None  None
8         content_updated_date        DATE  YES  None    None  None
9                 content_type     VARCHAR  YES  None    None  None
10               search_volume      BIGINT  YES  None    None  None
11                 competition      DOUBLE  YES  None    None  None
12           competition_level     VARCHAR  YES  None    None  None
13                         

In [7]:
max_date = pd.to_datetime(con.execute(f"SELECT MAX(report_date) FROM {TABLES['fact_daily']}").fetchone()[0])
last30_start = max_date - pd.Timedelta(days=30)
prev30_start = max_date - pd.Timedelta(days=60)

con.execute(f"""
CREATE OR REPLACE TABLE agg_cached AS
SELECT client_hash_id, content_hash_id,
    SUM(CASE WHEN report_date > DATE '{last30_start.date()}' THEN gsc_impressions ELSE 0 END) AS imp_last30,
    SUM(CASE WHEN report_date > DATE '{last30_start.date()}' THEN gsc_clicks ELSE 0 END) AS clk_last30,
    AVG(CASE WHEN report_date > DATE '{last30_start.date()}' THEN gsc_avg_position END) AS pos_last30,
    SUM(CASE WHEN report_date > DATE '{prev30_start.date()}' AND report_date <= DATE '{last30_start.date()}' THEN gsc_impressions ELSE 0 END) AS imp_prev30,
    SUM(CASE WHEN report_date > DATE '{prev30_start.date()}' AND report_date <= DATE '{last30_start.date()}' THEN gsc_clicks ELSE 0 END) AS clk_prev30,
    AVG(CASE WHEN report_date > DATE '{prev30_start.date()}' AND report_date <= DATE '{last30_start.date()}' THEN gsc_avg_position END) AS pos_prev30
FROM {TABLES['fact_daily']}
GROUP BY client_hash_id, content_hash_id
""")
print(con.execute("SELECT COUNT(*) FROM agg_cached").fetchone())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(427292,)


In [8]:
# Safety: reuse max_date if it's still in memory from the earlier cell;
# recompute cheaply only if the kernel lost it.
try:
    max_date
except NameError:
    max_date = pd.to_datetime(con.execute(f"SELECT MAX(report_date) FROM {TABLES['fact_daily']}").fetchone()[0])

query = f"""
SELECT
    a.*,
    cl.access_profile, cl.gsc_data_start, cl.ga4_data_start,
    cl.has_gsc_access, cl.has_ga4_access,
    dc.word_count, dc.char_count, dc.content_type, dc.main_intent,
    dc.content_created_date, dc.content_updated_date,
    DATE_DIFF('day', dc.content_updated_date, DATE '{max_date.date()}') AS days_since_last_update
FROM agg_cached a
JOIN {TABLES['dim_clients']} cl ON a.client_hash_id = cl.client_hash_id
LEFT JOIN {TABLES['dim_content']} dc ON a.content_hash_id = dc.content_hash_id
WHERE a.imp_prev30 >= 100
"""
feature_df = con.execute(query).fetchdf()
feature_df['ctr_prev30'] = feature_df['clk_prev30'] / feature_df['imp_prev30'] * 100
feature_df['is_declining_label'] = feature_df['imp_last30'] < 0.8 * feature_df['imp_prev30']

print(feature_df.shape)
print(feature_df[['word_count', 'content_type', 'main_intent', 'days_since_last_update']].isnull().sum())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(111247, 22)
word_count                24045
content_type                  0
main_intent                1873
days_since_last_update        0
dtype: int64


# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HassanNawaz14/FlyRank-ML-Internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [9]:
import numpy as np

# 1. Bucket days_since_last_update and print n and the is_declining_label rate per bucket.
print("--- Staleness Buckets and Decline Rate ---")

bins = [0, 90, 180, np.inf]
labels = ['<=90', '91-180', '181+']
feature_df['staleness_bucket'] = pd.cut(feature_df['days_since_last_update'], bins=bins, labels=labels, right=True)

staleness_summary = feature_df.groupby('staleness_bucket').agg(
    n=('client_hash_id', 'count'),
    is_declining_rate=('is_declining_label', lambda x: x.mean() * 100) # Percentage
).reset_index()
print(staleness_summary.to_string())

# 2. Compute position_tier_prev from pos_prev30 using exact cutoffs.
# Then compute median ctr_prev30 per position_tier_prev, and flag a page
# as weak CTR when its own ctr_prev30 is below its tier's median.
print("\n--- Position Tiers, Median CTR, and Weak CTR Flag ---")

pos_bins = [-np.inf, 3, 10, 20, 50, np.inf]
pos_labels = ['top_3', 'page_1', 'striking', 'page_3_5', 'deep']
feature_df['position_tier_prev'] = pd.cut(feature_df['pos_prev30'], bins=pos_bins, labels=pos_labels, right=True)

# Calculate median ctr_prev30 per position_tier_prev
median_ctr_by_tier = feature_df.groupby('position_tier_prev')['ctr_prev30'].median().rename('median_ctr_prev30').reset_index()

# Merge median CTR back to the main DataFrame
feature_df = feature_df.merge(median_ctr_by_tier, on='position_tier_prev', how='left')

# Flag weak CTR
feature_df['weak_ctr_flag'] = feature_df['ctr_prev30'] < feature_df['median_ctr_prev30']

# Summarize for printing
ctr_summary = feature_df.groupby('position_tier_prev').agg(
    n=('client_hash_id', 'count'),
    median_ctr_prev30=('ctr_prev30', 'median'),
    weak_ctr_rate=('weak_ctr_flag', lambda x: x.mean() * 100) # Percentage
).reset_index()
print(ctr_summary.to_string())

--- Staleness Buckets and Decline Rate ---
  staleness_bucket      n  is_declining_rate
0             <=90  76092          62.095884
1           91-180  15429          72.655389
2             181+    185          78.918919

--- Position Tiers, Median CTR, and Weak CTR Flag ---
  position_tier_prev      n  median_ctr_prev30  weak_ctr_rate
0              top_3    631           0.684151      49.920761
1             page_1  36195           0.320171      49.998619
2           striking  32917           0.207039      49.992405
3           page_3_5  35096           0.000000       0.000000
4               deep   6408           0.000000       0.000000


/tmp/ipykernel_13685/4201987734.py:10: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  staleness_summary = feature_df.groupby('staleness_bucket').agg(
/tmp/ipykernel_13685/4201987734.py:26: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  median_ctr_by_tier = feature_df.groupby('position_tier_prev')['ctr_prev30'].median().rename('median_ctr_prev30').reset_index()
/tmp/ipykernel_13685/4201987734.py:35: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and si

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [10]:
import numpy as np

# 1. Compute score (0-2) and reason_code per row.
# Staleness threshold: days_since_last_update > 90 (based on staleness_summary where decline rate jumps from ~62% to ~72%)
# Weak CTR threshold: ctr_prev30 < median_ctr_prev30 (already calculated as weak_ctr_flag)

feature_df['stale_flag'] = (feature_df['days_since_last_update'] > 90).astype(int)
feature_df['score'] = feature_df['stale_flag'] + feature_df['weak_ctr_flag'].astype(int)

def get_reason_code(row):
    if row['stale_flag'] and row['weak_ctr_flag']:
        return 'STALE+WEAK_CTR'
    elif row['stale_flag']:
        return 'STALE'
    elif row['weak_ctr_flag']:
        return 'WEAK_CTR'
    else:
        return 'NONE'

feature_df['reason_code'] = feature_df.apply(get_reason_code, axis=1)

# 2. Map score to action: 0 -> 'leave', 1 -> 'monitor', 2 -> 'refresh'.
score_to_action_map = {
    0: 'leave',
    1: 'monitor',
    2: 'refresh'
}
feature_df['action'] = feature_df['score'].map(score_to_action_map)

# 3. Rank rows by score descending, tie-broken by imp_prev30 descending.
ranked_df = feature_df.sort_values(by=['score', 'imp_prev30'], ascending=[False, False])

# 4. !mkdir -p work/outputs then write columns to work/outputs/baseline_action_score.csv.
output_dir = 'work/outputs'
os.makedirs(output_dir, exist_ok=True)

output_columns = [
    'content_hash_id',
    'client_hash_id',
    'score',
    'action',
    'reason_code',
    'days_since_last_update',
    'ctr_prev30',
    'position_tier_prev',
    'imp_prev30'
]

output_path = os.path.join(output_dir, 'baseline_action_score.csv')
ranked_df[output_columns].to_csv(output_path, index=False)

# 5. Print the row count written and the value_counts() of action.
print(f"\nCSV written to: {output_path}")
print(f"Total rows written: {len(ranked_df)}")
print("\nAction distribution:")
print(ranked_df['action'].value_counts().to_string())


CSV written to: work/outputs/baseline_action_score.csv
Total rows written: 111247

Action distribution:
action
leave      65114
monitor    41784
refresh     4349


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [11]:
refresh_picks = ranked_df[ranked_df['action'] == 'refresh']

# Calculate the fraction of 'refresh' picks that are declining
if len(refresh_picks) > 0:
    declining_refresh_picks_count = refresh_picks['is_declining_label'].sum()
    total_refresh_picks = len(refresh_picks)
    fraction_declining_refresh = declining_refresh_picks_count / total_refresh_picks
    print(f"Fraction of 'refresh' picks with is_declining_label == True: {fraction_declining_refresh:.2%}")
else:
    print("No 'refresh' picks found.")

print("\nColumns used to compute score in Section 1/2:")
scoring_columns = ['days_since_last_update', 'ctr_prev30', 'pos_prev30', 'position_tier_prev', 'imp_prev30']
for col in scoring_columns:
    print(f"- {col}")

Fraction of 'refresh' picks with is_declining_label == True: 74.80%

Columns used to compute score in Section 1/2:
- days_since_last_update
- ctr_prev30
- pos_prev30
- position_tier_prev
- imp_prev30


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


The rule, which combines staleness and weak CTR, is designed as a baseline to identify content potentially needing attention. The calculated match rate between 'refresh' picks and the `is_declining_label` being True provides an initial measure of the rule's alignment with observed decline.

It is important to note that `is_declining_label` was used *only* as a post-hoc check for evaluating the rule's performance, and was never an input to the rule itself. This ensures that the rule stands on its own merits, based on the defined heuristics.

Confirming no leakage, the rule's input columns are:
- `days_since_last_update` (content-metadata)
- `ctr_prev30` (prev30 performance metric derived from raw data)
- `pos_prev30` (prev30 performance metric derived from raw data)
- `position_tier_prev` (derived from `pos_prev30`)
- `imp_prev30` (prev30 performance metric derived from raw data)

None of the following were used as input for the rule: `trend_direction`, `trend_pct`, `imp_last30`, `clk_last30`, `pos_last30`, or any ID used directly as a feature. The inputs are either 'previous 30 days' aggregates/metadata or join keys, adhering to the principle of not using future or target-related information for the rule definition.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.